# Recovering Missing Boroughs with NYC Geosupport

This notebook is the third step in the project workflow. It recovers missing
boroughs through auditable evidence—precinct geography, local NYC Geosupport
functions, repeated camera descriptions, and a reviewed manual lookup—without
overwriting the original cleaned borough.

Only approved assignments are stored in `borough_recovery`. The
`parking_analysis` view exposes one final `analysis_borough` for downstream
work, while `borough_unresolved` preserves the small review set.

## 1. Setup

The default backend is the installed Geosupport Desktop Python library.
Set `GEOSUPPORT_BACKEND=api` in `.env` to use NYC Geoservice instead; that
backend also requires `GEOSERVICE_API_KEY`.

`REBUILD_LOOKUPS=False` reuses completed SQLite audit tables. On a fresh
database the notebook automatically creates them. Change it to `True` only
when the source data or resolution rules have changed.

In [1]:
from argparse import Namespace
from pathlib import Path
import os
import sqlite3
import sys

import pandas as pd
from dotenv import load_dotenv
from IPython.display import display


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError(
        "Could not find pyproject.toml. Start Jupyter inside the project folder."
    )


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

load_dotenv(PROJECT_ROOT / ".env")

RAW_FILE = PROJECT_ROOT / "data/raw/nycparking2025.csv"
DATABASE_FILE = PROJECT_ROOT / "data/database/nyc_parking.sqlite"
MANUAL_LOOKUP_FILE = PROJECT_ROOT / "data/reference/manual_borough_lookup.csv"
CACHE_FILE = PROJECT_ROOT / "data/processed/geosupport_cache.sqlite"

BACKEND = os.getenv("GEOSUPPORT_BACKEND", "local").strip().lower()
GEOSUPPORT_PATH = os.getenv("GEOSUPPORT_PATH", "")
REBUILD_LOOKUPS = False
LOOKUP_LIMIT = 0  # 0 means every eligible missing-borough row

if BACKEND not in {"local", "api"}:
    raise ValueError("GEOSUPPORT_BACKEND must be 'local' or 'api'.")

print(f"Project root: {PROJECT_ROOT}")
print(f"Geosupport backend: {BACKEND}")

Project root: C:\Users\jayson.coker\Documents\nyc-parking-analytics
Geosupport backend: local


**What this cell does:** Locates the project, loads environment
settings, imports the recovery modules, and selects the local Geosupport
binding by default.

### Validate reproducible inputs

The database is produced by notebook 02. The raw CSV is required only when
an audit table must be built or rebuilt. The manual lookup is intentionally
tracked in Git because it cannot be recreated from an API call.

In [2]:
required_files = [DATABASE_FILE, MANUAL_LOOKUP_FILE]
missing_files = [path for path in required_files if not path.exists()]
if missing_files:
    formatted = "\n".join(f"- {path}" for path in missing_files)
    raise FileNotFoundError(f"Required project files are missing:\n{formatted}")


def sqlite_object_exists(name: str, object_type: str = "table") -> bool:
    with sqlite3.connect(DATABASE_FILE) as connection:
        return bool(
            connection.execute(
                "SELECT 1 FROM sqlite_master WHERE type = ? AND name = ?",
                (object_type, name),
            ).fetchone()
        )


with sqlite3.connect(DATABASE_FILE) as connection:
    source_counts = pd.read_sql_query(
        '''
        SELECT
            COUNT(*) AS total_summons,
            SUM(CASE WHEN borough IS NULL OR TRIM(borough) = '' THEN 1 ELSE 0 END)
                AS originally_missing_borough
        FROM parking_violations
        ''',
        connection,
    )

display(source_counts)

,total_summons,originally_missing_borough
0,7056788,157778


**Result:** Both reproducible inputs exist. The database contains
7,056,788 tickets, including 157,778 whose original borough is missing.

## 2. Stage 1 — Precinct and street-code recovery

This stage reads the original street-code fields that were not retained in
the analytical table.

- Official NYPD violation precinct/location values are mapped to boroughs.
- Five-digit street codes are tested as borough-specific B5SC values.
- Geosupport validates candidates with display, address, intersection,
  segment, or street-stretch functions.
- A borough is accepted only when the evidence is unique or corroborated.
- Precinct/Geosupport conflicts remain in `review`.

Every attempted row is stored in `borough_geosupport_audit`; no stage CSV is
created.

In [3]:
from nycparking.geocoding.geosupport_boroughs import (
    DEFAULT_GEOSUPPORT_PATH,
    run as run_street_code_recovery,
)

STAGE1_TABLE = "borough_geosupport_audit"

if sqlite_object_exists(STAGE1_TABLE) and not REBUILD_LOOKUPS:
    with sqlite3.connect(DATABASE_FILE) as connection:
        stage1_audit = pd.read_sql_query(
            f"SELECT * FROM {STAGE1_TABLE}",
            connection,
        )
    print(f"Reused {len(stage1_audit):,} Stage 1 audit rows from SQLite.")
else:
    if not RAW_FILE.exists():
        raise FileNotFoundError(
            f"{RAW_FILE} is required to build Stage 1. "
            "Download the FY2025 source described in data/README.md."
        )
    stage1_args = Namespace(
        raw_file=RAW_FILE,
        output_file=None,
        cache_file=CACHE_FILE,
        backend=BACKEND,
        geosupport_path=GEOSUPPORT_PATH or str(DEFAULT_GEOSUPPORT_PATH),
        limit=LOOKUP_LIMIT,
        chunksize=250_000,
        delay=0.1,
        timeout=30.0,
        prepare_only=False,
    )
    stage1_audit = run_street_code_recovery(stage1_args)
    with sqlite3.connect(DATABASE_FILE) as connection:
        stage1_audit.to_sql(
            STAGE1_TABLE,
            connection,
            if_exists="replace",
            index=False,
        )
    print(f"Stored {len(stage1_audit):,} Stage 1 audit rows in SQLite.")

display(
    stage1_audit["status"]
    .value_counts(dropna=False)
    .rename_axis("status")
    .reset_index(name="summons_count")
)

Reused 22,466 Stage 1 audit rows from SQLite.


,status,summons_count
0,accepted,15887
1,review,3656
2,ambiguous,2841
3,unmatched,82


**Result:** Stage 1 audits 22,466 candidates and accepts 15,887 based
on precinct and Geosupport street-code/address evidence. Review, ambiguous,
and unmatched results remain unassigned.

### Stage 1 acceptance check

Only `accepted` rows with one of the five valid borough names can enter the
final recovery table. Ambiguous, review, and unmatched rows remain available
for audit but are not silently filled.

In [4]:
VALID_BOROUGHS = {
    "Bronx",
    "Brooklyn",
    "Manhattan",
    "Queens",
    "Staten Island",
}

stage1_accepted = stage1_audit.loc[
    stage1_audit["status"].eq("accepted")
    & stage1_audit["suggested_borough"].isin(VALID_BOROUGHS)
].copy()

print(f"Stage 1 accepted: {len(stage1_accepted):,}")
display(
    stage1_accepted["suggested_borough"]
    .value_counts()
    .rename_axis("borough")
    .reset_index(name="summons_count")
)

Stage 1 accepted: 15,887


,borough,summons_count
0,Manhattan,5151
1,Brooklyn,4395
2,Queens,3690
3,Bronx,2206
4,Staten Island,445


**Result:** The accepted Stage 1 assignments span all five boroughs.
Only approved statuses and valid borough names advance to consolidation.

## 3. Stage 2 — Camera-description recovery

Rows omitted from Stage 1 usually have zero street codes and precinct values
that are not standard NYPD precincts. Their `street_name` often contains a
camera description such as `SB WEBSTER AVE @ E 1`.

The resolver evaluates each unique normalized description once:

1. reuse an exact description only when known-borough evidence is unique;
2. validate parsed intersections with Geosupport Function 2;
3. validate a uniquely occurring street with Function 1N;
4. browse truncated cross-street names and recheck the intersection.

Results are expanded back to summons numbers and stored in
`borough_description_audit`.

In [5]:
from nycparking.geocoding.camera_description_resolver import (
    normalize_description,
    run as run_description_recovery,
)

STAGE2_TABLE = "borough_description_audit"

if sqlite_object_exists(STAGE2_TABLE) and not REBUILD_LOOKUPS:
    with sqlite3.connect(DATABASE_FILE) as connection:
        stage2_audit = pd.read_sql_query(
            f"SELECT * FROM {STAGE2_TABLE}",
            connection,
        )
    print(f"Reused {len(stage2_audit):,} Stage 2 audit rows from SQLite.")
else:
    stage2_args = Namespace(
        database_file=DATABASE_FILE,
        audit_data=stage1_audit,
        audit_file=None,
        lookup_file=None,
        matches_file=None,
        accepted_file=None,
        backend=BACKEND,
        geosupport_path=GEOSUPPORT_PATH or str(DEFAULT_GEOSUPPORT_PATH),
        cache_file=CACHE_FILE,
        limit=LOOKUP_LIMIT,
        delay=0.1,
        timeout=30.0,
    )
    _, stage2_audit = run_description_recovery(stage2_args)
    with sqlite3.connect(DATABASE_FILE) as connection:
        stage2_audit.to_sql(
            STAGE2_TABLE,
            connection,
            if_exists="replace",
            index=False,
        )
    print(f"Stored {len(stage2_audit):,} Stage 2 audit rows in SQLite.")

display(
    stage2_audit["status"]
    .value_counts(dropna=False)
    .rename_axis("status")
    .reset_index(name="summons_count")
)

Reused 135,533 Stage 2 audit rows from SQLite.


,status,summons_count
0,accepted,131907
1,ambiguous,3075
2,unmatched,496
3,review,55


**Result:** Stage 2 evaluates 135,533 description-based candidates
and accepts 131,907. Ambiguous, review, and unmatched results are retained as
audit evidence rather than forced into a borough.

## 4. Stage 3 — Reviewed manual descriptions

The remaining repeated or truncated descriptions were reviewed outside the
automated resolver. The tracked lookup contains one row per normalized
description, its borough, confidence, and a short justification.

Manual review has the highest priority. Any disagreement with an automated
accepted result is counted below and remains auditable in
`borough_recovery_candidates`.

In [6]:
manual_lookup = pd.read_csv(
    MANUAL_LOOKUP_FILE,
    dtype="string",
).fillna("")

if manual_lookup["normalized_description"].duplicated().any():
    duplicates = manual_lookup.loc[
        manual_lookup["normalized_description"].duplicated(keep=False)
    ]
    raise ValueError(
        "Manual lookup contains duplicate descriptions:\n"
        + duplicates.to_string(index=False)
    )
if not set(manual_lookup["borough"]).issubset(VALID_BOROUGHS):
    raise ValueError("Manual lookup contains an invalid borough.")

display(
    manual_lookup.groupby(["borough", "confidence"])
    .size()
    .rename("description_count")
    .reset_index()
)

,borough,confidence,description_count
0,Bronx,high,56
1,Bronx,low,2
2,Bronx,medium,1
3,Brooklyn,high,31
4,Brooklyn,low,3
5,Brooklyn,medium,5
6,Manhattan,high,52
7,Manhattan,medium,5
8,Queens,high,43
9,Queens,low,5


**Result:** The tracked manual lookup contains 208 reviewed
descriptions: 187 high-confidence, 11 medium-confidence, and 10 low-confidence
decisions across the five boroughs.

## 5. Consolidate approved assignments

The final table contains only rows whose original borough was missing.
Priority is explicit:

1. Stage 1 accepted result
2. Stage 2 accepted result
3. reviewed manual mapping

Sorting by this priority and keeping the last row makes the process
deterministic. All candidates are retained separately, so an override never
erases its competing evidence.

In [7]:
with sqlite3.connect(DATABASE_FILE) as connection:
    missing_locations = pd.read_sql_query(
        '''
        SELECT summons_number, street_name
        FROM parking_violations
        WHERE borough IS NULL OR TRIM(borough) = ''
        ''',
        connection,
    )

missing_locations["normalized_description"] = missing_locations[
    "street_name"
].map(normalize_description)
missing_ids = set(missing_locations["summons_number"])

stage1_candidates = stage1_accepted[
    ["summons_number", "suggested_borough", "confidence", "validation_method"]
].rename(
    columns={
        "suggested_borough": "recovered_borough",
        "validation_method": "recovery_method",
    }
)
stage1_candidates["source_file"] = STAGE1_TABLE
stage1_candidates["priority"] = 1

stage2_candidates = stage2_audit.loc[
    stage2_audit["status"].eq("accepted")
    & stage2_audit["suggested_borough"].isin(VALID_BOROUGHS),
    ["summons_number", "suggested_borough", "confidence", "resolution_method"],
].rename(
    columns={
        "suggested_borough": "recovered_borough",
        "resolution_method": "recovery_method",
    }
)
stage2_candidates["source_file"] = STAGE2_TABLE
stage2_candidates["priority"] = 2

manual_candidates = missing_locations.merge(
    manual_lookup,
    on="normalized_description",
    how="inner",
    validate="many_to_one",
)[["summons_number", "borough", "confidence"]].rename(
    columns={"borough": "recovered_borough"}
)
manual_candidates["recovery_method"] = "manual_description_review"
manual_candidates["source_file"] = MANUAL_LOOKUP_FILE.name
manual_candidates["priority"] = 3

recovery_candidates = pd.concat(
    [stage1_candidates, stage2_candidates, manual_candidates],
    ignore_index=True,
)
recovery_candidates["summons_number"] = pd.to_numeric(
    recovery_candidates["summons_number"],
    errors="coerce",
).astype("Int64")
recovery_candidates = recovery_candidates.loc[
    recovery_candidates["summons_number"].isin(missing_ids)
    & recovery_candidates["recovered_borough"].isin(VALID_BOROUGHS)
].dropna(subset=["summons_number"])
recovery_candidates["summons_number"] = recovery_candidates[
    "summons_number"
].astype("int64")

candidate_borough_counts = recovery_candidates.groupby("summons_number")[
    "recovered_borough"
].nunique()
conflicting_ids = set(
    candidate_borough_counts[candidate_borough_counts.gt(1)].index
)

recovery = (
    recovery_candidates.sort_values(["summons_number", "priority"])
    .drop_duplicates("summons_number", keep="last")
    [
        [
            "summons_number",
            "recovered_borough",
            "confidence",
            "recovery_method",
            "source_file",
        ]
    ]
    .sort_values("summons_number")
    .reset_index(drop=True)
)

print(f"Approved recovery rows: {len(recovery):,}")
print(f"Rows with disagreeing accepted/manual evidence: {len(conflicting_ids):,}")

Approved recovery rows: 157,364
Rows with disagreeing accepted/manual evidence: 37


**Result:** Consolidation produces 157,364 approved summons-level
recoveries. Thirty-seven records contain disagreeing evidence and are resolved
by the documented evidence priority.

### Inspect disagreements

This bounded table shows every source for the first few conflicts. The final
choice follows the documented priority above; all alternatives remain in
SQLite for review.

In [8]:
conflict_preview = recovery_candidates.loc[
    recovery_candidates["summons_number"].isin(conflicting_ids)
].sort_values(["summons_number", "priority"])

display(conflict_preview.head(50))

,summons_number,recovered_borough,confidence,recovery_method,source_file,priority
1595,1481003057,Queens,medium,function_d,borough_geosupport_audit,1
147970,1481003057,Brooklyn,high,manual_description_review,manual_borough_lookup.csv,3
2021,1485532474,Brooklyn,high,precinct,borough_geosupport_audit,1
147998,1485532474,Queens,high,manual_description_review,manual_borough_lookup.csv,3
2056,1485794511,Staten Island,high,precinct+address_1e,borough_geosupport_audit,1
148000,1485794511,Bronx,high,manual_description_review,manual_borough_lookup.csv,3
2708,1489084940,Brooklyn,high,precinct,borough_geosupport_audit,1
148037,1489084940,Bronx,high,manual_description_review,manual_borough_lookup.csv,3
2727,1489087000,Bronx,high,precinct+address_1e,borough_geosupport_audit,1
148038,1489087000,Manhattan,medium,manual_description_review,manual_borough_lookup.csv,3


**What this table shows:** The preview makes disagreements
transparent by displaying each competing borough, confidence, method, source,
and priority. The winning assignment is deterministic and auditable.

## 6. Save one recovery table and create analysis views

`borough_recovery` contains one approved result per summons.
`parking_analysis` preserves the original `borough` and adds:

- `analysis_borough`: original borough, otherwise recovered borough;
- `borough_source`: `original`, `recovered`, or `missing`;
- recovery confidence, method, and evidence source.

`borough_unresolved` exposes the remaining rows for review without creating
another CSV.

In [9]:
with sqlite3.connect(DATABASE_FILE) as connection:
    recovery_candidates.to_sql(
        "borough_recovery_candidates",
        connection,
        if_exists="replace",
        index=False,
    )
    recovery.to_sql(
        "borough_recovery",
        connection,
        if_exists="replace",
        index=False,
    )
    connection.executescript(
        '''
        CREATE UNIQUE INDEX IF NOT EXISTS idx_borough_recovery_summons
            ON borough_recovery (summons_number);

        DROP VIEW IF EXISTS borough_unresolved;
        DROP VIEW IF EXISTS parking_analysis;

        CREATE VIEW parking_analysis AS
        SELECT
            p.*,
            COALESCE(NULLIF(TRIM(p.borough), ''), r.recovered_borough)
                AS analysis_borough,
            CASE
                WHEN NULLIF(TRIM(p.borough), '') IS NOT NULL THEN 'original'
                WHEN r.recovered_borough IS NOT NULL THEN 'recovered'
                ELSE 'missing'
            END AS borough_source,
            r.confidence AS recovery_confidence,
            r.recovery_method,
            r.source_file AS recovery_source_file
        FROM parking_violations AS p
        LEFT JOIN borough_recovery AS r
            ON r.summons_number = p.summons_number;

        CREATE VIEW borough_unresolved AS
        SELECT
            p.summons_number,
            p.issue_date,
            p.violation_code,
            p.violation_precinct,
            p.issuer_precinct,
            p.street_name AS source_description,
            s1.status AS stage1_status,
            s1.confidence AS stage1_confidence,
            s1.validation_method AS stage1_method,
            s1.suggested_borough AS stage1_suggested_borough,
            s2.status AS stage2_status,
            s2.confidence AS stage2_confidence,
            s2.resolution_method AS stage2_method,
            s2.suggested_borough AS stage2_suggested_borough
        FROM parking_violations AS p
        LEFT JOIN borough_recovery AS r
            ON r.summons_number = p.summons_number
        LEFT JOIN borough_geosupport_audit AS s1
            ON s1.summons_number = p.summons_number
        LEFT JOIN borough_description_audit AS s2
            ON s2.summons_number = p.summons_number
        WHERE (p.borough IS NULL OR TRIM(p.borough) = '')
          AND r.summons_number IS NULL;
        '''
    )

print("Created borough_recovery and the parking_analysis/borough_unresolved views.")

Created borough_recovery and the parking_analysis/borough_unresolved views.


**Result:** One compact `borough_recovery` table and two analysis
views replace a collection of intermediate CSV files.

## 7. Validation

The row-count reconciliation must balance:

`original borough + recovered borough + still missing = total summons`

The assertions stop execution if a duplicate, invalid borough, row-count
change, or reconciliation error appears.

In [10]:
with sqlite3.connect(DATABASE_FILE) as connection:
    validation = pd.read_sql_query(
        '''
        SELECT
            COUNT(*) AS total_summons,
            SUM(CASE WHEN borough_source = 'original' THEN 1 ELSE 0 END)
                AS original_borough,
            SUM(CASE WHEN borough_source = 'recovered' THEN 1 ELSE 0 END)
                AS recovered_borough,
            SUM(CASE WHEN borough_source = 'missing' THEN 1 ELSE 0 END)
                AS still_missing,
            COUNT(DISTINCT summons_number) AS distinct_summons
        FROM parking_analysis
        ''',
        connection,
    )
    invalid_recoveries = connection.execute(
        '''
        SELECT COUNT(*)
        FROM borough_recovery
        WHERE recovered_borough NOT IN
            ('Bronx', 'Brooklyn', 'Manhattan', 'Queens', 'Staten Island')
        '''
    ).fetchone()[0]
    duplicate_recoveries = connection.execute(
        '''
        SELECT COUNT(*)
        FROM (
            SELECT summons_number
            FROM borough_recovery
            GROUP BY summons_number
            HAVING COUNT(*) > 1
        )
        '''
    ).fetchone()[0]

row = validation.iloc[0]
assert row["total_summons"] == row["distinct_summons"]
assert (
    row["original_borough"]
    + row["recovered_borough"]
    + row["still_missing"]
    == row["total_summons"]
)
assert invalid_recoveries == 0
assert duplicate_recoveries == 0

display(validation)
print("All reconciliation and uniqueness checks passed.")

,total_summons,original_borough,recovered_borough,still_missing,distinct_summons
0,7056788,6899010,157364,414,7056788


All reconciliation and uniqueness checks passed.


**Result:** Original (6,899,010), recovered (157,364), and unresolved
(414) rows sum exactly to 7,056,788. Summons uniqueness is also preserved.

### Recovery methods and remaining review set

These summaries explain where recovered boroughs came from and leave a
bounded, readable preview of unresolved summons. Query the
`borough_unresolved` view directly to inspect all remaining rows.

In [11]:
with sqlite3.connect(DATABASE_FILE) as connection:
    recovery_summary = pd.read_sql_query(
        '''
        SELECT
            source_file AS recovery_stage,
            recovery_method,
            confidence,
            COUNT(*) AS summons_count
        FROM borough_recovery
        GROUP BY source_file, recovery_method, confidence
        ORDER BY summons_count DESC
        ''',
        connection,
    )
    unresolved_preview = pd.read_sql_query(
        '''
        SELECT *
        FROM borough_unresolved
        ORDER BY
            CASE WHEN source_description IS NULL OR TRIM(source_description) = ''
                 THEN 1 ELSE 0 END,
            source_description,
            summons_number
        LIMIT 50
        ''',
        connection,
    )

display(recovery_summary)
display(unresolved_preview)

,recovery_stage,recovery_method,confidence,summons_count
0,borough_description_audit,unique_street_function_1n,medium,68990
1,borough_description_audit,known_exact_description,high,32393
2,manual_borough_lookup.csv,manual_description_review,high,31877
3,borough_geosupport_audit,precinct+address_1e,high,11083
4,borough_description_audit,browse_intersection_function_2,medium,6235
5,borough_description_audit,parsed_intersection_function_2,high,3216
6,borough_geosupport_audit,precinct,high,2344
7,borough_geosupport_audit,precinct+intersection_2,high,707
8,borough_geosupport_audit,address_1e,high,277
9,manual_borough_lookup.csv,manual_description_review,medium,147


,summons_number,issue_date,violation_code,violation_precinct,issuer_precinct,source_description,stage1_status,stage1_confidence,stage1_method,stage1_suggested_borough,stage2_status,stage2_confidence,stage2_method,stage2_suggested_borough
0,1497924698,2024-06-26,98,0,18,10,unmatched,NaN,NaN,NaN,NaN,None,NaN,None
1,1495656524,2024-07-25,46,0,288,116 ST,unmatched,NaN,NaN,NaN,NaN,None,NaN,None
2,1499468878,2024-11-09,40,0,28,116 WEST,unmatched,NaN,NaN,NaN,NaN,None,NaN,None
3,1493274831,2024-05-23,19,75,0,47TH ST,review,low,precinct_geosupport_conflict,Brooklyn,NaN,None,NaN,None
4,1495384512,2024-07-09,40,107,107,60 ST,review,low,precinct_geosupport_conflict,Queens,NaN,None,NaN,None
5,1496494910,2024-06-11,98,102,62,63 ST,review,low,precinct_geosupport_conflict,Queens,NaN,None,NaN,None
6,1497036800,2024-09-11,41,108,968,67TH STREET,review,low,precinct_geosupport_conflict,Queens,NaN,None,NaN,None
7,1498254858,2024-08-13,46,102,102,73 ST,review,low,precinct_geosupport_conflict,Queens,NaN,None,NaN,None
8,1494578074,2024-08-13,40,102,102,75 ST,review,low,precinct_geosupport_conflict,Queens,NaN,None,NaN,None
9,1494629185,2024-07-29,40,0,102,75 STREET,ambiguous,NaN,NaN,NaN,NaN,None,NaN,None


**Result:** The method summary documents how recoveries were made,
while `borough_unresolved` exposes the remaining 414 tickets for optional
future review.

## 8. Use the recovered borough in analysis

Later notebooks should query `parking_analysis`, not merge another CSV and
not overwrite the source borough. Use `analysis_borough` for grouping and
retain `borough_source` when an analysis needs to distinguish observed from
recovered values.

In [12]:
with sqlite3.connect(DATABASE_FILE) as connection:
    borough_counts = pd.read_sql_query(
        '''
        SELECT
            analysis_borough AS borough,
            borough_source,
            COUNT(*) AS summons_count
        FROM parking_analysis
        GROUP BY analysis_borough, borough_source
        ORDER BY borough, borough_source
        ''',
        connection,
    )

display(borough_counts)

,borough,borough_source,summons_count
0,NaN,missing,414
1,Bronx,original,968981
2,Bronx,recovered,110748
3,Brooklyn,original,1779380
4,Brooklyn,recovered,5232
5,Manhattan,original,1857772
6,Manhattan,recovered,32986
7,Queens,original,1992488
8,Queens,recovered,4814
9,Staten Island,original,300389


**Result:** The final grouped check shows original and recovered
borough counts together. Downstream analysis should use
`parking_analysis.analysis_borough` and `borough_source`.

## Takeaways

- The original cleaned data contained 157,778 tickets without a borough.
- The staged evidence workflow recovered 157,364 of them and left 414
  unresolved rather than assigning unsupported guesses.
- `summons_number` remains unique, and the reconciliation total equals all
  7,056,788 cleaned tickets.
- Approved assignments and unresolved evidence live in SQLite; the only
  tracked review artifact is `data/reference/manual_borough_lookup.csv`.
- Notebook 04 reads `parking_analysis`, so its borough charts automatically
  include these recoveries.